# 05 · Emission integration, F1, the pooled scalar channel

Runs the F1 grid, joint-trained BKT with the pooled misconception scalar as a second, additive emission channel beside slip. The chains are fitted once under the chain of record, strong snap, and shared frozen across every row. Correctness is the only joint-training target, the channel input m at turn k is the chains' filtered state before that turn, solution row consumed by the chains, never by BKT, evaluation is paper-aligned, each dialogue's first scored turn updates the filter and is excluded from metrics, unseen KCs contribute 0.5.

**The rows:**

- Validity, beta pinned to 0, a like-for-like BKT refit whose distance from the frozen M1, 60.65 accuracy, 64.28 AUC, 55.60 f1, certifies the engine before any delta is read.
- The full grid, every connection, mastered, unmastered, both, crossed with every pooling method, max, noisy_or, top2_or, mean, each under a free beta and beta pinned to 1, capture at full strength.

**Reading order.** The validity row first, then the connection and pooling comparisons in sections 3 and 4. Fitted betas read as lower bounds on the capture rates, attenuated by chain measurement noise.

## 1. Setup

Chains fitted once, shared by every row.

In [1]:
import pandas as pd
from scripts.load_data import load_paper_filtered_data
from scripts.chain import TriggerChain
from scripts.misconception_chains import MisconceptionChains
from scripts.emission_integration_pooled import (
    CHAIN_OF_RECORD,
    EmissionIntegrationPooledMastered,
    EmissionIntegrationPooledUnmastered,
    EmissionIntegrationPooledBoth,
)

train_df = load_paper_filtered_data("data/mathdial_train.csv")
test_df = load_paper_filtered_data("data/mathdial_test.csv")

chains = MisconceptionChains(train_df, test_df, chain_class=TriggerChain,
                             chain_kwargs=dict(CHAIN_OF_RECORD))
chains.run()
chains.summary()

,chain,pi,onset,resolve,pP_i,pP_a,expected_dwell_turns,accuracy,tpr,tnr,auc,f1,informative_cells
0,comprehension,0.05,0.0,0.0323,0.02,0.97,31.0,0.6327,0.6705,0.3704,0.5639,0.7614,1285
1,relevance,0.05,0.0,0.0535,0.02,0.97,18.7,0.6370,0.6061,0.7222,0.6749,0.7101,135
2,principles,0.05,0.0,0.0113,0.02,0.97,88.8,0.8784,0.7742,0.9535,0.8593,0.8421,74
3,wrong_operation,0.05,0.0,0.0881,0.02,0.97,11.3,0.7036,0.5512,0.8435,0.7511,0.6403,1164
4,steps,0.05,0.0,0.0475,0.02,0.97,21.0,0.8516,0.5580,0.9599,0.7217,0.6696,512


## 2. Models

The full grid, every connection crossed with every pooling method, each under a free beta and beta pinned to 1, plus the validity row. Twenty-five fits, loop-built so each row's configuration is its name. The emission is additive throughout, competing risks with disjoint causes, incorrect probability is slip plus beta times m on the wired branches, clipped at the floor, so where capture exceeds a branch's success mass the branch saturates to certain incorrectness.

In [2]:
CONNECTIONS = {
    "mastered": EmissionIntegrationPooledMastered,
    "unmastered": EmissionIntegrationPooledUnmastered,
    "both": EmissionIntegrationPooledBoth,
}
POOLINGS = ["max", "noisy_or", "top2_or", "mean"]

ROWS = {"validity, beta=0": (EmissionIntegrationPooledMastered,
                             {"pooling": "max", "pin_beta": 0.0})}
for connection, cls in CONNECTIONS.items():
    for pooling in POOLINGS:
        ROWS[f"{connection}, {pooling}, free"] = (cls, {"pooling": pooling})
        ROWS[f"{connection}, {pooling}, beta=1"] = (cls, {"pooling": pooling,
                                                          "pin_beta": 1.0})

models = {}
for name, (cls, kwargs) in ROWS.items():
    model = cls(train_df, test_df, chains=chains, **kwargs)
    model.run()
    models[name] = model
    print(f"{name:28s} {model.metrics}")

validity, beta=0             {'accuracy': 0.6035, 'auc': 0.6397, 'f1': 0.5531, 'turns': 1985, 'beta_mastered': 0.0, 'beta_unmastered': None, 'pooling': 'max', 'connection': 'mastered'}
mastered, max, free          {'accuracy': 0.6035, 'auc': 0.6423, 'f1': 0.5416, 'turns': 1985, 'beta_mastered': 0.178, 'beta_unmastered': None, 'pooling': 'max', 'connection': 'mastered'}
mastered, max, beta=1        {'accuracy': 0.6065, 'auc': 0.642, 'f1': 0.5803, 'turns': 1985, 'beta_mastered': 1.0, 'beta_unmastered': None, 'pooling': 'max', 'connection': 'mastered'}
mastered, noisy_or, free     {'accuracy': 0.6055, 'auc': 0.6431, 'f1': 0.544, 'turns': 1985, 'beta_mastered': 0.1759, 'beta_unmastered': None, 'pooling': 'noisy_or', 'connection': 'mastered'}
mastered, noisy_or, beta=1   {'accuracy': 0.6045, 'auc': 0.644, 'f1': 0.5809, 'turns': 1985, 'beta_mastered': 1.0, 'beta_unmastered': None, 'pooling': 'noisy_or', 'connection': 'mastered'}
mastered, top2_or, free      {'accuracy': 0.6055, 'auc': 0.643,

## 3. Results table

The frozen M1 row is the anchor, its numbers from notebook 02. Deltas are against it.

In [3]:
import numpy as np

M1 = {"accuracy": 0.6065, "auc": 0.6428, "f1": 0.5560}

results = pd.DataFrame(
    [{"row": name, **models[name].metrics} for name in ROWS])
for metric in ("accuracy", "auc", "f1"):
    results[f"d_{metric}"] = (results[metric] - M1[metric]).round(4)
results = results.set_index("row")
results[["connection", "pooling", "beta_mastered", "beta_unmastered",
         "accuracy", "d_accuracy", "auc", "d_auc", "f1", "d_f1", "turns"]]

auc_pivot = results.reset_index()
auc_pivot["beta_mode"] = np.where(auc_pivot["row"].str.contains("beta=1"),
                                  "pinned", "free")
auc_pivot = auc_pivot[auc_pivot["row"] != "validity, beta=0"]
auc_pivot.pivot_table(index=["connection", "pooling"],
                      columns="beta_mode", values="auc").round(4)

beta_mode              free  pinned
connection pooling                 
both       max       0.6326  0.6012
           mean      0.6438  0.6347
           noisy_or  0.6357  0.6005
           top2_or   0.6350  0.6030
mastered   max       0.6423  0.6420
           mean      0.6415  0.6361
           noisy_or  0.6431  0.6440
           top2_or   0.6430  0.6431
unmastered max       0.6373  0.6339
           mean      0.6432  0.6380
           noisy_or  0.6382  0.6354
           top2_or   0.6384  0.6342

## 4. Deltas against the engine baseline

Every model against the validity row's own three metrics, the engine's beta-pinned-to-zero BKT, so each delta isolates what the channel configuration changed with code path, protocol, and optimizer held fixed. Sorted by AUC delta.

In [4]:
baseline = models["validity, beta=0"].metrics
deltas = pd.DataFrame([
    {"row": name,
     "d_accuracy": round(models[name].metrics["accuracy"]
                         - baseline["accuracy"], 4),
     "d_auc": round(models[name].metrics["auc"] - baseline["auc"], 4),
     "d_f1": round(models[name].metrics["f1"] - baseline["f1"], 4)}
    for name in ROWS if name != "validity, beta=0"])
deltas.set_index("row").sort_values("d_auc", ascending=False)

,d_accuracy,d_auc,d_f1
row,,,
"mastered, noisy_or, beta=1",0.0010,0.0043,0.0278
"both, mean, free",0.0046,0.0041,-0.0033
"unmastered, mean, free",0.0036,0.0035,-0.0040
"mastered, noisy_or, free",0.0020,0.0034,-0.0091
"mastered, top2_or, beta=1",-0.0005,0.0034,0.0246
"mastered, top2_or, free",0.0020,0.0033,-0.0091
"mastered, max, free",0.0000,0.0026,-0.0115
"mastered, max, beta=1",0.0030,0.0023,0.0272
"mastered, mean, free",-0.0020,0.0018,-0.0059


## 5. The co-elevation contrast

Test turns with the strongest chain live, stratified by whether the second-strongest is also live, observed incorrect rates compared. Under max pooling the two groups carry the same pooled risk, under noisy_or the two-live group carries more, so the observed rates indicate which combination rule the corpus follows. Underpowered if co-elevated turns are few, in which case say so rather than overclaim.

In [5]:
import numpy as np
from scripts.pooling import Pooling

headline = models["mastered, max, free"]
outcomes = headline.predict(headline.test)[
    ["dialogue_id", "turn_index", "correct"]].rename(
    columns={"turn_index": "position"})

records = []
for d, track in chains.predict_states(test_df).items():
    strongest = Pooling.max(track)
    second = Pooling.second_max(track)
    for pos in range(1, len(strongest)):
        records.append({"dialogue_id": d, "position": pos,
                        "strongest": strongest[pos], "second": second[pos]})
elev = pd.DataFrame(records).merge(outcomes, on=["dialogue_id", "position"])
elev = elev[elev["strongest"] >= 0.5]
elev["group"] = np.where(elev["second"] >= 0.5, "two live", "one live")

elev.groupby("group").agg(
    turns=("correct", "size"),
    incorrect_rate=("correct", lambda c: round(1 - c.mean(), 4)),
    mean_strongest=("strongest", "mean"),
).round(4)

,turns,incorrect_rate,mean_strongest
group,,,
one live,1892,0.5518,0.8309
two live,278,0.6007,0.8676


## 6. Interpretation ledger

- The validity row's distance from M1 bounds what initialization and the optimizer contribute, read every other delta net of it.
- The fitted betas are per-branch capture rates, fitted independently where both branches are wired, beta_mastered's gap from 0 is the channel's measured strength on the mastered branch, and attenuation from chain noise makes each a lower bound. On the both rows the split between the two betas locates where capture acts.
- If the unmastered rows match or trail the baseline while the mastered rows lead it, the channel's contribution is competence-side.
- Slip attribution, compare each KC's fitted slip here against the validity row's, the drop is how much of the baseline's slip the channel re-attributed, same optimizer both sides.

In [6]:
m2 = models["mastered, max, free"]
bkt = models["validity, beta=0"]
slips = pd.DataFrame([
    {"kc": kc, "m2_slip": round(p["slip"], 4),
     "bkt_slip": round(bkt.parameters[kc]["slip"], 4)}
    for kc, p in m2.parameters.items()])
slips["drop"] = (slips["bkt_slip"] - slips["m2_slip"]).round(4)
print(f"median slip drop {slips['drop'].median():+.4f} over {len(slips)} KCs, "
      f"beta_mastered {m2.metrics['beta_mastered']}")
slips.sort_values("drop", ascending=False).head(10)

median slip drop +0.0482 over 138 KCs, beta_mastered 0.178


,kc,m2_slip,bkt_slip,drop
1,"Add and subtract within 1000, using concrete m...",0.0001,0.5531,0.5530
134,Use variables to represent quantities in a rea...,0.0001,0.3788,0.3787
19,Compare two fractions with different numerator...,0.6965,0.9999,0.3034
82,Recognize and represent proportional relations...,0.0562,0.3371,0.2809
65,Interpret multiplication as scaling (resizing)...,0.2310,0.5000,0.2690
23,"Count to 120, starting at any number less than...",0.1657,0.4301,0.2644
118,Understand the concept of a unit rate a/b asso...,0.2988,0.5064,0.2076
51,Find whole-number quotients of whole numbers w...,0.5795,0.7850,0.2055
132,Use the structure of an expression to identify...,0.0001,0.2008,0.2007
97,Solve systems of linear equations exactly and ...,0.4275,0.6000,0.1725


## 7. Why the channel did not improve prediction

The grid's verdict is a null on the primary metric, the best row sits 0.4 AUC points over the engine baseline and dead even with frozen M1, inside the 0.3-point band that initialization and optimizer alone are worth here, and the mechanism is readable off the run's own numbers.

**The channel measures persistence, the metric rewards timing.** The filtered m is high across whole thread spans, a belief that stays active keeps m near 0.9 through every turn of its thread, including the locally-correct ones, and the probe layer found A-cells inside live threads running 79 to 93 per cent correct. So m separates struggling dialogues from clean ones but barely separates the incorrect turns from the correct turns inside a struggling dialogue, and within-dialogue turn ranking is most of what the AUC can still pay for.

**Joint training lets mastery absorb the same signal first.** A dialogue with many wrong answers gets a low mastery estimate from correctness history alone, and mastery already gates the emission, so the channel is paid only for what it adds beyond that history. The fitted beta_mastered of 0.178 is exactly that residual, small not because beliefs rarely cause errors, the same-turn annotation tables say they overwhelmingly do, but because the causal, before-the-turn state adds little the correctness stream had not already implied.

**What did move, and why it is threshold, not ranking.** The beta-pinned-to-one mastered rows hold AUC while lifting f1 by 0.02 to 0.028, full-strength capture pushes captured turns below 0.5, correcting the baseline's shortage of incorrect predictions. The ordering of predictions barely changes, their calibration around the threshold does.

**The rest of the grid behaves as designed.** The unmastered rows trail the baseline, the channel's contribution is competence-side. The poolings are indistinguishable, max, noisy_or, and top2 within 0.002, because off-seed chains sit at their floors and the pooled scalar is effectively the strongest chain everywhere. The co-elevation contrast leans toward compounding, incorrect rates of 0.601 with two live chains against 0.552 with one, 278 co-elevated turns, but the two-live group also carries a slightly stronger strongest chain, so the contrast is suggestive, not decisive. And the slip attribution shows the re-attribution is real but concentrated, a median drop of 0.048 with top-KC drops of 0.2 to 0.55, movement in the parameter ledger that does not translate into held-out ranking.

## 8. Notes

- Both sides of every comparison are paper-aligned by construction, per-KC predictions averaged to true turns, each dialogue's first scored turn updating the filter but excluded from metrics, exactly 0.5 classifying to 0, unseen KCs contributing 0.5, the notebook 02 protocol.
- The slip attribution in section 6 compares against the validity row's slips, same optimizer, only the channel differs, so the drop is purely the channel's re-attribution.
- Every model keeps its fitted parameters at `.parameters` and the shared frozen chains at `.chains`, nothing here refits the chains.
- The winner's row name, pooling, connection, and beta get logged with the pick, and the F2 and F3 faces reuse this notebook's shape.